**فصل سوم: تولید دستیارهای هوشمند صوتی**

**درس 8:**

# 🧩پروژه ۲: ساخت سیستم بیدارباش

### 1️⃣ نصب و وارد کردن ابزارها

In [ ]:
%pip install -v -i https://mirror-pypi.runflare.com/simple pyaudio numpy librosa scipy pygame --trusted-host mirror-pypi.runflare.com --quiet

In [1]:
import pyaudio
import numpy as np
import librosa
import time
from scipy.spatial.distance import cdist
import warnings
import pygame
import os
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore')

pygame 2.6.1 (SDL 2.28.4, Python 3.12.2)
Hello from the pygame community. https://www.pygame.org/contribute.html


### 2️⃣ کالیبراسیون هوشمند: گوش دادن به سکوت

In [2]:
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024

def get_audio_stream():
    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                    input=True, frames_per_buffer=CHUNK)
    return p, stream

In [3]:
def calibrate_threshold(duration=1.0):
    p, stream = get_audio_stream()
    print(f"کالیبراسیون نویز ({duration} ثانیه)\n لطفاً ساکت باشید")
    frames = []
    for _ in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK, exception_on_overflow=False)
        frames.append(np.frombuffer(data, dtype=np.int16))
    stream.stop_stream()
    stream.close()
    p.terminate()
    noise = np.concatenate(frames)
    rms = np.sqrt(np.mean(noise.astype(np.float32)**2))
    print(f"نویز محیط: {rms:.2f}")
    return rms

In [5]:
NOISE_RMS = calibrate_threshold(1.0)
ENERGY_THRESHOLD = max(NOISE_RMS * 2.0, 1.0)   
SILENCE_CHUNKS = 20               # 20 چانک سکوت ≈ 1.28 ثانیه
MAX_WAIT_CHUNKS = int((3.0 * RATE) / CHUNK)   # حداکثر 3 ثانیه انتظار برای شروع


کالیبراسیون نویز (1.0 ثانیه)
 لطفاً ساکت باشید
نویز محیط: 64.84


### 3️⃣ (VAD) تشخیص فعالیت صوتی 

In [6]:
def record_utterance(timeout=4.0):
    p, stream = get_audio_stream()
    print("گوش می‌دم... (منتظر صدا)")
    frames = []
    silence_count = 0
    started = False
    waited = 0
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        signal = np.frombuffer(data, dtype=np.int16)
        rms = np.sqrt(np.mean(signal.astype(np.float32)**2))

        if not started:
            waited += 1
            if waited > MAX_WAIT_CHUNKS:
                print("زمان انتظار برای صدا تمام شد")
                break
        
        if rms > ENERGY_THRESHOLD:
            if not started:
                print("🔊 صدا شنیده شد، در حال ضبط")
            started = True
            silence_count = 0
            frames.append(data)
        elif started:
            silence_count += 1
            frames.append(data)
            if silence_count >= SILENCE_CHUNKS:
                print("🔇 سکوت تشخیص داده شد، پایان ضبط")
                break

    stream.stop_stream()
    stream.close()
    p.terminate()
    audio = np.frombuffer(b''.join(frames), dtype=np.int16).astype(np.float32) / 32768.0
    audio, _ = librosa.effects.trim(audio, top_db=20)
    return audio

### 4️⃣ استخراج اثر انگشت پیشرفته 

In [7]:
def extract_mfcc(audio, sr=RATE, n_mfcc=13):
    # MFCC پایه
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=160)
    # دلتا و دلتا-دلتا
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    full_mfcc = np.vstack([mfcc, mfcc_delta, mfcc_delta2]) 

    full_mfcc = (full_mfcc - full_mfcc.mean(axis=1, keepdims=True)) / (full_mfcc.std(axis=1, keepdims=True) + 1e-8)
    return full_mfcc

### 5️⃣ آموزش کلمه بیدارباش

In [10]:
templates = []
print("سه بار پشت سر هم عبارت انتخابی را بگویید. بعد از هر بار، ۲ ثانیه فرصت دارید")
for i in range(3):
    print(f"\n--- ضبط الگوی {i+1} ---")
    time.sleep(1.5)   # کمی فرصت آماده شدن
    audio = record_utterance(timeout=4.0)
    if len(audio) < 100:   # اگر صدا خیلی کوتاه بود (احتمالاً timeout شده)
        print("صدایی ضبط نشد. دوباره تلاش کنید")
        continue
    mfcc = extract_mfcc(audio)
    templates.append(mfcc)
    print(f"✔ الگوی {i+1} ضبط شد. (ابعاد: {mfcc.shape})")
    time.sleep(2.0)   # فاصله تا ضبط بعدی

if len(templates) < 3:
    print("تعداد الگوها کافی نیست، لطفاً سلول را دوباره اجرا کنید")
else:
    print("✅ هر ۳ الگو با موفقیت ذخیره شدند")

سه بار پشت سر هم عبارت انتخابی را بگویید. بعد از هر بار، ۲ ثانیه فرصت دارید

--- ضبط الگوی 1 ---
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
✔ الگوی 1 ضبط شد. (ابعاد: (39, 93))

--- ضبط الگوی 2 ---
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
✔ الگوی 2 ضبط شد. (ابعاد: (39, 109))

--- ضبط الگوی 3 ---
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
✔ الگوی 3 ضبط شد. (ابعاد: (39, 90))
✅ هر ۳ الگو با موفقیت ذخیره شدند


### 6️⃣ (DTW) منطق تطبیق با کشش زمانی 

In [11]:
def dtw_distance(x, y):
    cost = cdist(x.T, y.T, metric='cosine')   # ماتریس هزینه
    D, wp = librosa.sequence.dtw(C=cost, subseq=False)
    return D[-1, -1] / len(wp)                 # نرمال‌سازی بر طول مسیر

def match_keyword(test_vec, template_list, threshold):
    distances = [dtw_distance(test_vec, t) for t in template_list]
    
    min_dist = min(distances)
    return min_dist < threshold, min_dist

### 7️⃣ پیدا کردن عدد جادویی 

In [12]:
print("حالا ۳ عبارت متفاوت بگویید (مثلاً: «خوبم»، «چه خبر»، و سکوت):")
for i in range(3):
    audio = record_utterance()
    mfcc = extract_mfcc(audio)
    dists = [dtw_distance(mfcc, t) for t in templates]
    print(f"کمترین فاصله با الگو: {min(dists):.3f}")
    
THRESHOLD = 0.60      # عدد جادویی ما
print(f"✅ آستانه روی {THRESHOLD} تنظیم شد.")

حالا ۳ عبارت متفاوت بگویید (مثلاً: «خوبم»، «چه خبر»، و سکوت):
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
کمترین فاصله با الگو: 0.833
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
کمترین فاصله با الگو: 0.760
گوش می‌دم... (منتظر صدا)
🔊 صدا شنیده شد، در حال ضبط
🔇 سکوت تشخیص داده شد، پایان ضبط
کمترین فاصله با الگو: 0.764
✅ آستانه روی 0.6 تنظیم شد.


### 8️⃣ افزودن خلاقیت های صوتی و بصری

In [13]:
def show_assistant_alert_vscode(audio_file="alert.wav", duration=7):
    clear_output(wait=True)
    
    try:
        if not os.path.exists(audio_file):
            print(f"فایل صوتی پیدا نشد. مطمئن شوید '{audio_file}' در پوشه پروژه است")
        else:
            if not pygame.mixer.get_init():
                pygame.mixer.init()
            
            sound = pygame.mixer.Sound(audio_file)
            sound.play()
    except Exception as e:
        print(f"خطای سیستم در پخش صدا: {e}")

    html_design = """
    <div style="display: flex; flex-direction: column; align-items: center; justify-content: center; height: 350px; background-color: transparent; font-family: 'Courier New', monospace; overflow: hidden;">
        
        <h2 style="color: #00f0ff; text-shadow: 0 0 15px #00f0ff; margin-bottom: 50px; text-align: center; letter-spacing: 2px;">
            [ SYSTEM ACTIVATED ]<br><br>
            <span style="font-size: 0.6em; color: #a5f3fc; letter-spacing: 1px;">...آماده ی فرمان بعدی هستم...</span>
        </h2>
        
        <div class="ai-core-outer">
            <div class="ai-core-inner"></div>
        </div>
        
        <style>
            .ai-core-outer {
                width: 120px; 
                height: 120px; 
                border-radius: 50%; 
                border: 3px solid transparent;
                border-top: 3px solid #00f0ff;
                border-bottom: 3px solid #00f0ff;
                box-shadow: 0 0 30px rgba(0, 240, 255, 0.4);
                position: relative;
                animation: spin 2s linear infinite, pulse 1.5s ease-in-out infinite alternate;
                display: flex;
                align-items: center;
                justify-content: center;
            }
            
            .ai-core-inner {
                width: 80px;
                height: 80px;
                border-radius: 50%;
                border: 2px dashed #00f0ff;
                box-shadow: inset 0 0 15px rgba(0, 240, 255, 0.5);
                animation: spin-reverse 4s linear infinite;
            }

            @keyframes spin { 
                0% { transform: rotate(0deg); }
                100% { transform: rotate(360deg); } 
            }
            @keyframes spin-reverse { 
                0% { transform: rotate(360deg); }
                100% { transform: rotate(0deg); } 
            }
            @keyframes pulse {
                0% { box-shadow: 0 0 20px rgba(0, 240, 255, 0.3); }
                100% { box-shadow: 0 0 60px rgba(0, 240, 255, 0.8); }
            }
        </style>
    </div>
    """
    
    display(HTML(html_design))
    
    # توقف برای نمایش انیمیشن و پخش صدا
    time.sleep(duration)
    
    # پاکسازی نهایی
    clear_output(wait=True)
    print("زمان گوش دادن به اتمام رسید. سیستم خاموش شد")


### 9️⃣ راه‌اندازی هسته هوش مصنوعی 

In [14]:

ALERT_FILE = "alert.wav" 
THRESHOLD = 0.60

clear_output(wait=True)
print("✨سیستم بیدار و منتظر فرمان است✨\n")

# روشن کردن اولیه میکسر قبل از ورود به حلقه
if not pygame.mixer.get_init():
    pygame.mixer.init()

try:
    while True:
        test_audio = record_utterance()
        if len(test_audio) < 512:
            continue

        test_mfcc = extract_mfcc(test_audio)
        detected, dist = match_keyword(test_mfcc, templates, THRESHOLD)

        if detected:
            show_assistant_alert_vscode(audio_file=ALERT_FILE, duration=20)
            break
        
        time.sleep(0.1)

except KeyboardInterrupt:
    print("پردازش دستی متوقف شد")

زمان گوش دادن به اتمام رسید. سیستم خاموش شد


### 🏁 جمع‌بندی



**امروز چه کارهایی کردیم؟**

1. **جریان زنده صدا** را با اتصال مستقیم به میکروفون دریافت کردیم
2. **سیستم تشخیص فعالیت صوتی** ساختیم تا سیستم، سکوت و نویز را از صدای انسان تشخیص دهد
3. **اثر انگشت پیشرفته صدا** را با ترکیب یک فیچر مهم و مشتق‌های آن استخراج کردیم
4. **الگوریتم جدیدی** را پیاده‌سازی کردیم تا سیستم، کلمه کلیدی را حتی اگر تند یا کِش‌دار ادا شود تشخیص دهد
5. **هسته همیشه بیدار** دستیار صوتی‌مان را با یک رابط کاربری جذاب تست و روشن کردیم

!شما حالا یک سیستم بیدارباش اختصاصی زنده ساختید

با توجه به اینکه دستیارهای هوشمندی مثل سیری و الکسا هم دقیقاً با همین منطق همیشه به گوش هستند، از حالا، شما هسته اولیه یک دستیار صوتی هوشمند را در سیستم خود دارید

**:قدم بعدی** 

در ویدیوی آینده سراغ مدل قدرتمندی می‌رویم تا سیستم، تمام حرف‌هایمان را بعد از بیدار شدن کلمه به کلمه تایپ کند!